# Energy Consumption Regression Analysis

This notebook analyzes the energy consumption of various energy types based on the average output tokens per prompt using polynomial regression models. The steps involve loading the data, transforming it, fitting the regression models, predicting values, and visualizing the results.


## 1. Load Data

First, we load the data from a CSV file into a pandas DataFrame.

In [1]:
import pandas as pd
from sklearn.linear_model import LinearRegression
import numpy as np
import matplotlib.pyplot as plt

import altair as alt

from IPython.display import display, Math

In [2]:
# Function to load CSV data
def load_csv_data(file_path):
    """
    Load data from a CSV file into a pandas DataFrame.
    
    Parameters:
        file_path (str): The path to the CSV file.
    
    Returns:
        pd.DataFrame: The loaded DataFrame.
    """
    df = pd.read_csv(file_path)
    return df

In [3]:
# Specify the path to your CSV file
file_path = 'data/params_test.csv'

# Load the data into a DataFrame
df_vllm_emission_regression = load_csv_data(file_path)

# Display the first few rows of the DataFrame
df_vllm_emission_regression

,model_setup,parameters,num_gpus,num_prompts,total_time,time_per_prompt,tok_per_sec,total_out_tok,total_in_tok,avg_out_tok,avg_in_tok,emissions_per_1M_prompts,total_energy_per_1M_prompts,cpu_energy_per_1M_prompts,gpu_energy_per_1M_prompts,ram_energy_per_1M_prompts,idle_gpu_energy_per_1M_prompts,non_idle_gpu_energy_per_1M_prompts
0,70B_8GPUs,70,8,7500,5364.228628,0.715230,215.326206,1155059.0,1905000.0,154.007867,254.0,115.968001,174.401537,10.301362,98.036465,66.063711,44.503230,53.533235
1,34B_8GPUs,34,8,7500,2930.255841,0.390701,334.550651,980319.0,1920000.0,130.709200,256.0,60.124206,90.419374,5.627253,48.702914,36.089207,24.310271,24.392643
2,13B_8GPUs,13,8,7500,1550.814195,0.206775,554.112158,859325.0,1920000.0,114.576667,256.0,31.845256,47.891329,2.978204,25.817401,19.095723,12.866014,12.951387
3,7B_8GPUs,7,8,7500,1074.694161,0.143293,921.779456,990631.0,1920000.0,132.084133,256.0,21.879479,32.904032,2.063889,17.602871,13.237272,8.915981,8.686889
4,34B_4GPUs,34,4,7500,2291.938375,0.305592,427.230073,979185.0,1920000.0,130.558000,256.0,25.314835,38.070383,4.401405,26.617137,7.051840,9.507300,17.109837
5,13B_4GPUs,13,4,7500,1333.532095,0.177804,648.803282,865200.0,1920000.0,115.360000,256.0,14.795574,22.250715,2.560907,15.586729,4.103079,5.531689,10.055040
6,7B_4GPUs,7,4,7500,936.629993,0.124884,1057.411152,990403.0,1920000.0,132.053733,256.0,10.085866,15.167895,1.798727,10.487412,2.881756,3.885280,6.602132
7,7B_1GPUs,7,1,7500,2837.988473,0.378398,348.690282,989579.0,1920000.0,131.943867,256.0,11.665944,17.544138,5.449993,9.187937,2.906207,2.943099,6.244838


## 2. Data Transformation

Convert energy values from kilowatt-hours (kWh) to watt-hours (Wh) for better granularity, and calculate prompts per second.

In [4]:
# Transform energy values from kWh to Wh
df_vllm_emission_regression['total_energy_7500_prompts_Wh'] = df_vllm_emission_regression['total_energy_per_1M_prompts'] / 1000000 * 7500 * 1000
df_vllm_emission_regression['ram_energy_7500_prompts_Wh'] = df_vllm_emission_regression['ram_energy_per_1M_prompts'] / 1000000 * 7500 * 1000
df_vllm_emission_regression['gpu_energy_7500_prompts_Wh'] = df_vllm_emission_regression['gpu_energy_per_1M_prompts'] / 1000000 * 7500 * 1000
df_vllm_emission_regression['cpu_energy_7500_prompts_Wh'] = df_vllm_emission_regression['cpu_energy_per_1M_prompts'] / 1000000 * 7500 * 1000
df_vllm_emission_regression['gpu_idle_energy_7500_prompts_Wh'] = df_vllm_emission_regression['idle_gpu_energy_per_1M_prompts'] / 1000000 * 7500 * 1000
df_vllm_emission_regression['gpu_non_idle_energy_7500_prompts_Wh'] = df_vllm_emission_regression['non_idle_gpu_energy_per_1M_prompts'] / 1000000 * 7500 * 1000
df_vllm_emission_regression['prompt_per_sec'] = df_vllm_emission_regression['num_prompts'] / df_vllm_emission_regression['total_time']
df_vllm_emission_regression['model_type'] = 'Code LLaMA'

df_vllm_emission_regression = df_vllm_emission_regression[['model_setup', 
                                                           'model_type', 
                                                           'parameters',
                                                           'num_gpus',
                                                           'num_prompts', 
                                                           'total_time', 
                                                           'prompt_per_sec', 
                                                           'total_out_tok', 
                                                           'total_in_tok', 
                                                           'avg_out_tok', 
                                                           'avg_in_tok', 
                                                           'total_energy_7500_prompts_Wh', 
                                                           'ram_energy_7500_prompts_Wh', 
                                                           'gpu_energy_7500_prompts_Wh', 
                                                           'cpu_energy_7500_prompts_Wh',
                                                           'gpu_idle_energy_7500_prompts_Wh', 
                                                           'gpu_non_idle_energy_7500_prompts_Wh']]

smallest_setup = [
    '7B_1GPUs', '13B_4GPUs', '34B_4GPUs', '70B_8GPUs'
]

largest_setup = [
    '70B_8GPUs', '34B_8GPUs', '13B_8GPUs', '7B_8GPUs'
]

lowest_energy_setup = [
    '7B_4GPUs', '13B_4GPUs', '34B_4GPUs', '70B_8GPUs'
]

df_smallest_setup = df_vllm_emission_regression[df_vllm_emission_regression['model_setup'].isin(smallest_setup)]
df_largest_setup = df_vllm_emission_regression[df_vllm_emission_regression['model_setup'].isin(largest_setup)]
df_lowest_energy_setup = df_vllm_emission_regression[df_vllm_emission_regression['model_setup'].isin(lowest_energy_setup)]

df_dict = {
    'smallest_setup': df_smallest_setup,
    'largest_setup': df_largest_setup,
    'lowest_energy_setup': df_lowest_energy_setup
}

print("="*20 + " Smallest Setup " + "="*20)
print(df_dict['smallest_setup']['model_setup'])

print("\n")

print("="*20 + " Largest Setup " + "="*20)
print(df_dict['largest_setup']['model_setup'])

print("\n")

print("="*20 + " Lowest Energy Setup " + "="*20)
print(df_dict['lowest_energy_setup']['model_setup'])


==================== Smallest Setup ====================
0    70B_8GPUs
4    34B_4GPUs
5    13B_4GPUs
7     7B_1GPUs
Name: model_setup, dtype: object


==================== Largest Setup ====================
0    70B_8GPUs
1    34B_8GPUs
2    13B_8GPUs
3     7B_8GPUs
Name: model_setup, dtype: object


==================== Lowest Energy Setup ====================
0    70B_8GPUs
4    34B_4GPUs
5    13B_4GPUs
6     7B_4GPUs
Name: model_setup, dtype: object


## 3. Polynomial Regression Model Fitting
Fit polynomial regression models for each type of energy consumption using the average output tokens per prompt as the feature variable.

In [5]:
def fit_regression(X, y, regression_type='linear'):
    if regression_type == 'linear':
        model = LinearRegression()
    elif regression_type == 'exponential':
        # Transform y for exponential regression
        y = np.log(y)
        model = LinearRegression()
    else:
        raise ValueError("Unsupported regression type")
    
    model.fit(X, y)
    return model

In [6]:
# Define the feature variable
X = df_smallest_setup[['parameters']]

In [7]:
# Fit models for each energy consumption type
models = {}
energy_types = [
    'total_energy_7500_prompts_Wh', 
    'ram_energy_7500_prompts_Wh',
    'gpu_energy_7500_prompts_Wh',
    'cpu_energy_7500_prompts_Wh',
    'gpu_idle_energy_7500_prompts_Wh',
    'gpu_non_idle_energy_7500_prompts_Wh'
]

In [8]:
for setup, df in df_dict.items():
    for energy_type in energy_types:
        model_name = f'{setup}___{energy_type}'

        y = df[energy_type]

        if setup == 'largest_setup': 
            models[model_name] = fit_regression(X, y, regression_type='linear')
        else: 
            models[model_name] = fit_regression(X, y, regression_type='exponential')
        
        coefs = models[model_name].coef_
        intercept = models[model_name].intercept_

## 4. Display Model Coefficients
Display the coefficients of the polynomial regression models for each type of energy consumption.

In [9]:
def display_model_coefficients(model, model_name, regression_type='linear'):
    coefs = model.coef_
    intercept = model.intercept_

    # Format the coefficients to 4 decimal places for readability
    # coefs = np.round(coefs, 5)
    # intercept = np.round(intercept, 5)
    
    print("="*20 + f" Regression for {model_name} " + "="*20 + "\n")
    
    # Print raw coefficients to check their values
    print(f"Raw coefficients:\n intercept={intercept}, coefs={coefs}\n")
    
    print("Formula:")
    if regression_type == 'linear':
        # Generate the LaTeX formula for linear regression
        latex_formula = (
            f"\\hat{{y}} = {intercept:.5f} + {coefs[0]:.5f} x"
        )
    elif regression_type == 'exponential':
        # Generate the LaTeX formula for exponential regression
        latex_formula = (
            f"\\hat{{y}} = e^{{{intercept:.5f} + {coefs[0]:.5f} x}}"
        )
    else:
        raise ValueError("Unsupported regression type")

    # Display the LaTeX formula
    display(Math(latex_formula))

    print("\n\n")

In [10]:
# Display coefficients for each energy consumption type
for model_name, model in models.items():
    if 'largest_setup' in model_name:
        regression_type = 'linear'
    else:
        regression_type = 'exponential'
    
    display_model_coefficients(models[model_name], model_name, regression_type)

==================== Regression for smallest_setup___total_energy_7500_prompts_Wh ====================

Raw coefficients:
 intercept=4.589149053215258, coefs=[0.0360557]

Formula:


<IPython.core.display.Math object>




==================== Regression for smallest_setup___ram_energy_7500_prompts_Wh ====================

Raw coefficients:
 intercept=2.661625600771611, coefs=[0.04867423]

Formula:


<IPython.core.display.Math object>




==================== Regression for smallest_setup___gpu_energy_7500_prompts_Wh ====================

Raw coefficients:
 intercept=4.125483714756282, coefs=[0.03539422]

Formula:


...

<IPython.core.display.Math object>




==================== Regression for lowest_energy_setup___gpu_idle_energy_7500_prompts_Wh ====================

Raw coefficients:
 intercept=3.127447579047605, coefs=[0.03762203]

Formula:


<IPython.core.display.Math object>




==================== Regression for lowest_energy_setup___gpu_non_idle_energy_7500_prompts_Wh ====================

Raw coefficients:
 intercept=3.7876277959919116, coefs=[0.03164939]

Formula:


<IPython.core.display.Math object>

## 5. Predict Values
Define x values for prediction and predict the corresponding energy consumption values using the fitted models.

In [11]:
# Define the x values for prediction
predicted_values = {'parameters': [7, 10, 25, 45, 50, 70, 75]}

x_values = pd.DataFrame(predicted_values)

predicted_values_dfs = {}

for setup, df in df_dict.items():

    predicted_values_setup = predicted_values

    for energy_type in energy_types:
        model_name = f'{setup}___{energy_type}'

        model = models[model_name]

        if setup == 'largest_setup': 
            predicted_values_setup[energy_type] = model.predict(x_values)
        else:
            predicted_values_setup[energy_type] = np.exp(model.predict(x_values))
    
    predicted_values_dfs[setup] = pd.DataFrame(predicted_values_setup)

predicted_values_dfs['lowest_energy_setup']

,parameters,total_energy_7500_prompts_Wh,ram_energy_7500_prompts_Wh,gpu_energy_7500_prompts_Wh,cpu_energy_7500_prompts_Wh,gpu_idle_energy_7500_prompts_Wh,gpu_non_idle_energy_7500_prompts_Wh
0,7,117.996641,20.050022,84.577696,15.095261,29.689748,55.101310
1,10,132.044042,23.208144,93.684591,16.344189,33.237138,60.589503
2,25,231.720598,48.224559,156.217513,24.320754,58.439867,97.403529
3,45,490.485022,127.871000,308.895651,41.316982,124.019511,183.433126
4,50,591.617521,163.172892,366.296115,47.170066,149.687389,214.883730
5,70,1252.281996,432.665044,724.293165,80.134224,317.662543,404.675217
6,75,1510.488468,552.112728,858.884778,91.486270,383.408032,474.058978


## 6. Combine Actual and Predicted Data
Combine the actual and predicted data into a single DataFrame for visualization.

In [12]:
combined_dfs = {}

for setup, df in df_dict.items():
    # Combine actual and predicted data into a single DataFrame
    data = []

    predicted_values_setup = predicted_values

    for energy_type in energy_types:
        model_name = f'{setup}___{energy_type}'

        for index, row in df.iterrows():
            data.append({'parameters': row['parameters'], 'Energy_Consumption': row[energy_type], 'Type': 'Actual', 'Energy_Type': energy_type})
        for i, x in enumerate(x_values['parameters']):
            data.append({'parameters': x, 'Energy_Consumption': predicted_values_dfs[setup][energy_type][i], 'Type': 'Predicted', 'Energy_Type': energy_type})

    combined_dfs[setup] = pd.DataFrame(data)

combined_dfs['lowest_energy_setup']

,parameters,Energy_Consumption,Type,Energy_Type
0,70,1308.011531,Actual,total_energy_7500_prompts_Wh
1,34,285.527873,Actual,total_energy_7500_prompts_Wh
2,13,166.880362,Actual,total_energy_7500_prompts_Wh
3,7,113.759213,Actual,total_energy_7500_prompts_Wh
4,7,117.996641,Predicted,total_energy_7500_prompts_Wh
...,...,...,...,...
61,25,97.403529,Predicted,gpu_non_idle_energy_7500_prompts_Wh
62,45,183.433126,Predicted,gpu_non_idle_energy_7500_prompts_Wh
63,50,214.883730,Predicted,gpu_non_idle_energy_7500_prompts_Wh
64,70,404.675217,Predicted,gpu_non_idle_energy_7500_prompts_Wh


## 7. Visualize Results
Create an Altair chart to visualize the actual and predicted energy consumption values.

In [13]:
def visualize_chart(combined_df, type):
    # Create Altair chart
    base = alt.Chart(combined_df[combined_df['Type'] == 'Actual']).mark_point(size=100, filled=True).encode(
        x=alt.X('parameters', title='Parameters in Billion'),
        y=alt.Y('Energy_Consumption', title='Energy Consumption (Wh)'),
        color=alt.Color('Energy_Type', title='Energy Type'),
        tooltip=['parameters', 'Energy_Consumption', 'Energy_Type', 'Type']
    ).properties(
        width=1200,
        height=600
    )


    # Highlight predicted values
    predicted = alt.Chart(combined_df[combined_df['Type'] == 'Predicted']).mark_point(size=10, filled=False).encode(
        x=alt.X('parameters', title='Parameters in Billion'),
        y=alt.Y('Energy_Consumption', title='Energy Consumption (Wh)'),
        color=alt.Color('Energy_Type', title='Energy Type'),
        tooltip=['parameters', 'Energy_Consumption', 'Energy_Type']
    )

    if type == 'largest_setup':
        regression = predicted.transform_regression('parameters', 'Energy_Consumption', groupby=['Energy_Type'], method='linear').mark_line()  # method = "exp"
    else:
        regression = predicted.transform_regression('parameters', 'Energy_Consumption', groupby=['Energy_Type'], method='exp').mark_line()

    # Combine charts
    final_chart = base + regression + predicted

    return final_chart

In [14]:
for setup, df in combined_dfs.items():
    print("\n")
    print("="*50 + f" {setup} " + "="*50 + "\n")
    visualize_chart(df, setup).display()
    print("\n")
    print("\n")



================================================== smallest_setup ==================================================



alt.LayerChart(...)







================================================== largest_setup ==================================================



alt.LayerChart(...)







================================================== lowest_energy_setup ==================================================



alt.LayerChart(...)

In [15]:
# Extract the first row where parameters is 7
initial_row = df_largest_setup[df_largest_setup['parameters'] == 7].iloc[0]

# Create a new DataFrame with the required columns
theoretical_df = pd.DataFrame(columns=['parameters', 'predicted_energy'])

# Populate the new DataFrame
multipliers = [1, 2, 4, 8, 10]
for multiplier in multipliers:
    new_row = {
        'parameters': initial_row['parameters'] * multiplier,
        'predicted_energy': initial_row['total_energy_7500_prompts_Wh'] * multiplier
    }

    theoretical_df= pd.concat([theoretical_df, pd.DataFrame([new_row])]).reset_index(drop=True)

theoretical_df

,parameters,predicted_energy
0,7,246.780242
1,14,493.560485
2,28,987.120969
3,56,1974.241939
4,70,2467.802424


In [16]:
theoretical_model = fit_regression(theoretical_df[['parameters']], theoretical_df['predicted_energy'], regression_type='linear')


In [17]:
theoretical_vs_actual_df = pd.DataFrame(columns=['parameters', 'energy', 'type'])

theoretical_vs_actual_df = pd.concat([theoretical_vs_actual_df, theoretical_df.rename(columns={'predicted_energy': 'energy'}).assign(type='Theoretical')]).reset_index(drop=True)

theoretical_vs_actual_df

,parameters,energy,type
0,7,246.780242,Theoretical
1,14,493.560485,Theoretical
2,28,987.120969,Theoretical
3,56,1974.241939,Theoretical
4,70,2467.802424,Theoretical


In [18]:
theoretical_vs_actual_df = pd.concat([theoretical_vs_actual_df, df_largest_setup[['parameters', 'total_energy_7500_prompts_Wh']].rename(columns={'total_energy_7500_prompts_Wh': 'energy'}).assign(type='Actual')]).reset_index(drop=True)
theoretical_vs_actual_df = pd.concat([theoretical_vs_actual_df, df_lowest_energy_setup[['parameters', 'total_energy_7500_prompts_Wh']].rename(columns={'total_energy_7500_prompts_Wh': 'energy'}).assign(type='Actual_Lowest_Energy')]).reset_index(drop=True)

theoretical_vs_actual_df

,parameters,energy,type
0,7,246.780242,Theoretical
1,14,493.560485,Theoretical
2,28,987.120969,Theoretical
3,56,1974.241939,Theoretical
4,70,2467.802424,Theoretical
5,70,1308.011531,Actual
6,34,678.145303,Actual
7,13,359.184964,Actual
8,7,246.780242,Actual
9,70,1308.011531,Actual_Lowest_Energy


In [19]:
# Create Altair chart for theoretical and actual values
theoretical = alt.Chart(theoretical_vs_actual_df[theoretical_vs_actual_df['type']=='Theoretical']).mark_point(size=20, filled=True, opacity=0.3).encode(
    x=alt.X('parameters', title='Parameters'),
    y=alt.Y('energy', title='Energy Consumption (Wh)'),
    color=alt.Color('type', title='Type'),
    tooltip=['parameters', 'energy', 'type']
).properties(
    width=800,
    height=400
)

actual = alt.Chart(theoretical_vs_actual_df[theoretical_vs_actual_df['type']=='Actual']).mark_point(size=120, filled=True).encode(
    x=alt.X('parameters', title='Parameters'),
    y=alt.Y('energy', title='Energy Consumption (Wh)'),
    color=alt.Color('type', title='Type'),
    tooltip=['parameters', 'energy', 'type']
).properties(
    width=800,
    height=400
)

actual_lowest_energy = alt.Chart(theoretical_vs_actual_df[theoretical_vs_actual_df['type']=='Actual_Lowest_Energy']).mark_point(size=120, filled=True).encode(
    x=alt.X('parameters', title='Parameters'),
    y=alt.Y('energy', title='Energy Consumption (Wh)'),
    color=alt.Color('type', title='Type'),
    tooltip=['parameters', 'energy', 'type']
).properties(
    width=800,
    height=400
)    

# Create linear regression line
regression_theoretical = theoretical.transform_regression(
    'parameters', 'energy', groupby=['type'], method='linear'
).mark_line(strokeDash = [8, 8])

# Create linear regression line
regression_actual = actual.transform_regression(
    'parameters', 'energy', groupby=['type'], method='linear'
).mark_line()

regression_actual_lowest_energy = actual_lowest_energy.transform_regression(
    'parameters', 'energy', groupby=['type'], method='exp'
).mark_line()

# Combine charts
final_chart = actual + theoretical + actual_lowest_energy + regression_theoretical + regression_actual + regression_actual_lowest_energy

# Display the chart
final_chart

alt.LayerChart(...)